<a href="https://colab.research.google.com/github/Anmolsharma44/infosys-AI-powered-career-Assistant/blob/main/Milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Step 1: Install required system packages for OCR & Python libraries
!apt-get install -y poppler-utils tesseract-ocr > /dev/null
!pip install -q streamlit pyngrok pypdf google-generativeai pdfplumber pdf2image pytesseract pillow

# Step 2: Create the enhanced app.py file supporting PDFs and Scanned Images/OCR
app_code = """
import streamlit as st
import json
import google.generativeai as genai
from pypdf import PdfReader
import pdfplumber
from pdf2image import convert_from_bytes
import pytesseract
from PIL import Image
import io
import sqlite3

st.set_page_config(page_title="AI Internship Assistant", layout="centered")

st.title("🚀 AI-Powered Internship & Career Assistant")
st.subheader("Milestone 1: Smart Resume Parser (Supports PDF & Scanned Images)")

api_key = st.text_input("Enter your Gemini API Key", type="password")

# File uploader supporting both PDFs and Image formats
uploaded_file = st.file_uploader("Upload your Resume (PDF, PNG, JPG, JPEG)", type=["pdf", "png", "jpg", "jpeg"])

if uploaded_file and api_key:
    genai.configure(api_key=api_key)

    with st.spinner("Extracting text and parsing with AI (OCR enabled)..."):
        file_bytes = uploaded_file.read()
        extracted_text = ""
        file_extension = uploaded_file.name.split('.')[-1].lower()

        try:
            # Case 1: If uploaded file is an Image (PNG/JPG)
            if file_extension in ["png", "jpg", "jpeg"]:
                image = Image.open(io.BytesIO(file_bytes))
                extracted_text = pytesseract.image_to_string(image)

            # Case 2: If uploaded file is a PDF
            elif file_extension == "pdf":
                # Try standard text extraction first using pdfplumber
                with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
                    for page in pdf.pages:
                        text = page.extract_text()
                        if text:
                            extracted_text += text + "\\n"

                # If standard text extraction yields very little text, use OCR (Scanned PDF)
                if len(extracted_text.strip()) < 50:
                    images = convert_from_bytes(file_bytes)
                    for img in images:
                        ocr_text = pytesseract.image_to_string(img)
                        extracted_text += ocr_text + "\\n"

            # Check if text was successfully extracted
            if not extracted_text.strip():
                st.error("Could not extract any text from the uploaded file. Please ensure it is clear and readable.")
            else:
                st.info(f"Successfully extracted {len(extracted_text)} characters from the document.")

                # Gemini AI Parsing
                model = genai.GenerativeModel('gemini-4-hydrogen')
                prompt = f\"\"\"
                You are an expert AI Resume Parser. Analyze the resume text provided below and extract the information strictly into a valid JSON format:
                {{
                  "full_name": "Candidate's full name",
                  "email": "Email address",
                  "phone": "Phone number",
                  "skills": ["Skill 1", "Skill 2"],
                  "education": [{{"degree": "Degree name", "institution": "College", "year": "Year"}}],
                  "projects": [{{"title": "Project title", "description": "Description"}}]
                }}

                Resume Text:
                {extracted_text}
                \"\"\"

                response = model.generate_content(prompt)
                cleaned = response.text.replace("```json", "").replace("```", "").strip()

                data = json.loads(cleaned)
                st.success("Resume successfully parsed by Gemini AI!")
                st.json(data)

                # Save parsed profile to SQLite database
                conn = sqlite3.connect('candidate_database.db')
                cursor = conn.cursor()
                cursor.execute('''
                    CREATE TABLE IF NOT EXISTS candidates (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        full_name TEXT, email TEXT, phone TEXT, skills TEXT, education TEXT, projects TEXT
                    )
                ''')
                cursor.execute('INSERT INTO candidates (full_name, email, phone, skills, education, projects) VALUES (?, ?, ?, ?, ?, ?)',
                               (data.get('full_name'), data.get('email'), data.get('phone'),
                                json.dumps(data.get('skills')), json.dumps(data.get('education')), json.dumps(data.get('projects'))))
                conn.commit()
                conn.close()

                st.success("Profile successfully saved to SQLite database!")

        except Exception as e:
            st.error(f"An error occurred: {e}")
else:
    st.info("Please enter your API key and upload a resume (PDF or Image) to get started.")
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("Enhanced app.py with OCR support successfully created inside Colab!")

Enhanced app.py with OCR support successfully created inside Colab!


In [11]:
# Cloudflare tunnel download aur run karne ke liye
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!streamlit run app.py & ./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-11-14T16:56:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-14T16:56:04Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-11-14 16:56:06.612 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.80.211.202:8501

2026-11-14T16:56:08Z INF +-------------------------------------------------------------